# DCGAN para generar objetos de CIFAR-10

Este cuaderno implementa una DCGAN con convoluciones transpuestas en el generador y convoluciones con *stride* en el discriminador. Se usa CIFAR-10 disponible desde torchvision para una demostración educativa; las imágenes son datos de objetos de CIFAR-10 y se emplean solo para generación sintética, nunca para reconocer identidades. Ejecutar en Colab con GPU; el primer bloque puede tardar por la descarga del conjunto. La configuración corta prioriza reproducibilidad sobre realismo.

**Propósito:** observar el entrenamiento adversarial y guardar imágenes, pérdidas y un checkpoint. No se espera que una corrida breve produzca objetos de CIFAR-10 fotorrealistas.

In [ ]:
!pip -q install torch torchvision matplotlib scikit-learn


In [ ]:
import os, random, numpy as np, torch, torchvision
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt

SEED = 73
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device, '| PyTorch:', torch.__version__)


## 1. Datos y preprocesamiento

CIFAR-10 contiene imágenes de diez categorías de objetos y animales. Se selecciona un subconjunto fijo de hasta 5.000 imágenes y se normalizan los canales RGB a [-1, 1], que coincide con la salida `tanh` del generador. La partición es solo para entrenamiento no supervisado; las etiquetas de identidad no entran al modelo.

In [ ]:
image_size = 64
transform = transforms.Compose([transforms.Resize(image_size), transforms.CenterCrop(image_size), transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
dataset = CIFAR10(root='/content/data', train=True, download=True, transform=transform)
limit = min(5000, len(dataset))
indices = np.random.permutation(len(dataset))[:limit]
dataset = Subset(dataset, indices.tolist())
loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
print('Imágenes usadas:', len(dataset), '| lotes/época:', len(loader))
real, _ = next(iter(loader))
plt.figure(figsize=(8,4)); plt.axis('off'); plt.imshow(make_grid(real[:16], normalize=True, nrow=8).permute(1,2,0)); plt.show()


## 2. Arquitectura

El generador proyecta un vector latente de 100 dimensiones a un bloque espacial y duplica resolución mediante `ConvTranspose2d`. El discriminador reduce la imagen con convoluciones estriadas. Batch normalization y LeakyReLU siguen la receta DCGAN clásica; se usa salida logística binaria para la pérdida adversarial.

In [ ]:
nz, ngf, ndf, nc = 100, 64, 64, 3
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False), nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False), nn.Tanh())
    def forward(self, z): return self.net(z)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False), nn.Sigmoid())
    def forward(self, x): return self.net(x).view(-1)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
G, D = Generator().to(device), Discriminator().to(device)
G.apply(init_weights); D.apply(init_weights)
criterion = nn.BCELoss()
optG = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optD = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, nz, 1, 1, device=device)


## 3. Entrenamiento

Para reducir tiempo en Colab se proponen 3 épocas por defecto. Al terminar cada época se guarda una cuadrícula con el mismo ruido fijo. Las pérdidas son señales del juego entre redes; por sí solas no demuestran calidad, diversidad ni ausencia de sesgo.

In [ ]:
epochs = 3
loss_G, loss_D, snapshots = [], [], []
real_label, fake_label = 1.0, 0.0
os.makedirs('/content/resultados_dcgan_cifar10', exist_ok=True)
for epoch in range(epochs):
    sum_g = sum_d = 0.0
    for real_batch, _ in loader:
        real_batch = real_batch.to(device, non_blocking=True); b = real_batch.size(0)
        # Actualización del discriminador con imágenes reales y generadas
        D.zero_grad(set_to_none=True)
        labels = torch.full((b,), real_label, device=device)
        err_real = criterion(D(real_batch), labels); err_real.backward()
        z = torch.randn(b, nz, 1, 1, device=device); fake = G(z)
        labels.fill_(fake_label)
        err_fake = criterion(D(fake.detach()), labels); err_fake.backward()
        err_d = err_real + err_fake; optD.step()
        # Actualización del generador: busca que D etiquete sus salidas como reales
        G.zero_grad(set_to_none=True); labels.fill_(real_label)
        err_g = criterion(D(fake), labels); err_g.backward(); optG.step()
        sum_g += err_g.item(); sum_d += err_d.item()
    loss_G.append(sum_g/len(loader)); loss_D.append(sum_d/len(loader))
    with torch.no_grad(): sample = G(fixed_noise).detach().cpu()
    snapshots.append(sample); save_image(sample, f'/content/resultados_dcgan_cifar10/epoca_{epoch+1}.png', normalize=True, nrow=8)
    print(f'Época {epoch+1}/{epochs} | D={loss_D[-1]:.3f} | G={loss_G[-1]:.3f}')


## 4. Resultados y evidencias

La cuadrícula muestra la evolución con el mismo ruido inicial; la gráfica de pérdidas documenta el proceso. Evalúe textura, variedad y señales de repetición visualmente. Una pérdida oscilante es habitual en el entrenamiento adversarial.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,3.5)); ax[0].plot(loss_D, marker='o'); ax[0].set_title('Pérdida del discriminador'); ax[1].plot(loss_G, marker='o', color='tab:orange'); ax[1].set_title('Pérdida del generador');
for a in ax: a.set_xlabel('Época'); a.grid(alpha=.2)
plt.tight_layout(); plt.savefig('/content/resultados_dcgan_cifar10/perdidas.png', dpi=160); plt.show()
fig, axes = plt.subplots(1, epochs, figsize=(4*epochs,4))
for i, ax in enumerate(np.array(axes).reshape(-1)):
    ax.imshow(make_grid(snapshots[i][:16], normalize=True, nrow=4).permute(1,2,0)); ax.set_title(f'Época {i+1}'); ax.axis('off')
plt.tight_layout(); plt.savefig('/content/resultados_dcgan_cifar10/evolucion.png', dpi=160); plt.show()
torch.save({'generator':G.state_dict(),'discriminator':D.state_dict(),'loss_G':loss_G,'loss_D':loss_D}, '/content/resultados_dcgan_cifar10/dcgan_cifar10.pt')
print('Artefactos guardados en /content/resultados_dcgan_cifar10. Descárguelos para adjuntar evidencias.')


## 5. Lectura crítica

La salida no es una identificación ni una fotografía de una persona concreta: es una muestra sintética aproximada de patrones del conjunto. Una corrida de tres épocas y hasta 5.000 imágenes es exploratoria. Para comparar configuraciones conviene repetir con semillas distintas y revisar diversidad; no usar objetos de CIFAR-10 generados para inferir identidad, atributos sensibles o decisiones sobre clases.

## Referencias

Goodfellow, I., et al. (2014). Generative adversarial nets. *Advances in Neural Information Processing Systems, 27*. https://papers.nips.cc/paper/5423-generative-adversarial-nets

Mohr, F. (2017, 4 de noviembre). Implementing a generative adversarial network (GAN/DCGAN) to draw human faces. *Towards Data Science*. https://medium.com/data-science/implementing-a-generative-adversarial-network-gan-dcgan-to-draw-human-faces-8291616904a

PyTorch. (s. f.). *DCGAN tutorial*. https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html